# Data Preprocessing for Medical QA

This notebook prepares the PubMedQA dataset for transformer model training.

Steps performed:
1. Load dataset
2. Combine question and context
3. Encode labels
4. Tokenize text
5. Remove unnecessary columns
6. Train/test split
7. Convert dataset to PyTorch tensors

### Loading Required Libraries

In [28]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


### What this means

my device is  MacBook Air M4 and it supports Apple Metal GPU acceleration.

## 1. Import Required Libraries

In [29]:
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd

## 2. Check Hardware Environment

In [30]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


## 3. Load the PubMedQA Dataset

In [31]:
dataset = load_dataset("pubmed_qa", "pqa_labeled")

dataset

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

### My training pipeline will look like

Dataset
   ->
Tokenized data
    ->
Model
   ->
GPU (MPS)
   ->
Training

## 4. Combine Question and Context

In [32]:
def combine_text(example):
    context = " ".join(example["context"]["contexts"])
    example["input_text"] = "Question: " + example["question"] + " Context: " + context
    return example

dataset = dataset.map(combine_text)

### Verifying the Result

In [34]:
dataset["train"][0]["input_text"][:300]

'Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of th'

### Current Project Progress

### Project Pipeline

| Stage | Status |
|------|------|
| Dataset exploration | ✔ Completed |
| Data preprocessing | ⚙ Current stage |
| Tokenization | Upcoming |
| Baseline model training | Upcoming |
| RAG model implementation | Upcoming |
| Model comparison | Upcoming |
| Error analysis | Upcoming |
| Paper writing | Upcoming |

## 5. Encode Answer Labels

In [35]:
label_map = {"yes": 0, "no": 1, "maybe": 2}

def encode_labels(example):
    example["label"] = label_map[example["final_decision"]]
    return example

dataset = dataset.map(encode_labels)

### Verify Labels

In [36]:
dataset["train"][0]["final_decision"]

'yes'

### Why Numeric Labels Are Required
Deep learning models learn using mathematical loss functions.

The model predicts probabilities like:
[0.72, 0.20, 0.08]

Which means:

yes = 72%
no = 20%
maybe = 8%

## 6. Tokenize Text Using BERT Tokenizer

### Tokenization

adding the tokenizer:

In [37]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Tokenize the dataset:

In [38]:
def tokenize(example):
    return tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize)

### Verify Tokenization

In [39]:
tokenized_dataset["train"][0] 

{'pubid': 21645374,
 'question': 'Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?',
 'context': {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.',
   'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), ce

In [40]:
tokenized_dataset["train"][0].keys()

dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision', 'input_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

## 7. Remove Unnecessary Columns

In [41]:
tokenized_dataset = tokenized_dataset.remove_columns([
    "pubid",
    "question",
    "context",
    "long_answer",
    "final_decision",
    "input_text"
])

In [42]:
tokenized_dataset["train"][0].keys()

dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [43]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})

## 8. Create Train/Test Split

In [44]:
split_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

## 9. Convert Dataset to PyTorch Format

In [45]:
split_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [47]:
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [48]:
split_dataset["train"][0]

{'label': tensor(0),
 'input_ids': tensor([  101,  3160,  1024,  2003,  2045,  1037, 11658,  1999,  1996, 11394,
          2740,  1997,  2047, 15024,  2000,  1996,  2329,  4273,  2749,  1029,
          6123,  1024,  4481,  2013,  1996,  2329,  4721, 11394,  2578,  7487,
          2008,  3529,  5073,  1999,  1996,  2329,  2390,  2031,  1037, 14516,
          2135,  2896,  2504,  1997, 11394, 10516,  2084,  2216,  1999,  1996,
          2548,  3212,  2030,  1996,  2548,  2250,  2486,  1012,  2053,  2470,
          2018,  2042, 10607,  2000,  2004, 17119, 18249,  2065,  2023, 11138,
          1996,  8700,  2740,  1997, 15024,  5241,  2169,  2326,  1012,  2023,
          2817,  6461,  2000,  4405,  1037,  2832,  2005,  9334, 11394,  1998,
         17522,  3207,  5302, 14773,  2951,  2013,  2047, 15024,  2000,  2169,
          2326,  1998, 11628,  1996, 19701, 10744,  2008,  2053,  5966,  1999,
         11394,  2740,  5839,  1012, 16474,  9181,  2020,  2764,  1010,  1037,
          7099,  2

### Final Verification Cell

In [49]:
print(split_dataset)

print(split_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 800
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 200
    })
})
{'label': tensor(0), 'input_ids': tensor([  101,  3160,  1024,  2003,  2045,  1037, 11658,  1999,  1996, 11394,
         2740,  1997,  2047, 15024,  2000,  1996,  2329,  4273,  2749,  1029,
         6123,  1024,  4481,  2013,  1996,  2329,  4721, 11394,  2578,  7487,
         2008,  3529,  5073,  1999,  1996,  2329,  2390,  2031,  1037, 14516,
         2135,  2896,  2504,  1997, 11394, 10516,  2084,  2216,  1999,  1996,
         2548,  3212,  2030,  1996,  2548,  2250,  2486,  1012,  2053,  2470,
         2018,  2042, 10607,  2000,  2004, 17119, 18249,  2065,  2023, 11138,
         1996,  8700,  2740,  1997, 15024,  5241,  2169,  2326,  1012,  2023,
         2817,  6461,  2000,  4405,  1037,  2832,  2005,  9334, 11394,

In [50]:
split_dataset.save_to_disk("../data/processed/pubmedqa_processed")

Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

## Preprocessing Summary

The PubMedQA dataset was processed for transformer training.

Processing steps included:
- Combining question and context into a single input sequence
- Encoding answer labels (yes/no/maybe) into numeric format
- Tokenizing text using the BERT tokenizer
- Padding/truncating sequences to a maximum length of 512 tokens
- Splitting the dataset into training (80%) and testing (20%) subsets
- Converting dataset fields into PyTorch tensors

The processed dataset was saved for use in the model training stage.

In [51]:
dataset.save_to_disk("data/processed/pubmedqa")

Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

In [53]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'input_text', 'label'],
        num_rows: 800
    })
    test: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision', 'input_text', 'label'],
        num_rows: 200
    })
})
